# Least Squares Regression
In an earlier section, we developed formulas for the slope and intercept of the regression line through a *football shaped* scatter diagram. It turns out that the slope and intercept of the least squares line have the same formulas as those we developed, *regardless of the shape of the scatter plot*.

We saw this in the example about Little Women, but let's confirm it in an example where the scatter plot clearly isn't football shaped. For the data, we are once again indebted to the rich [data archive of Prof. Larry Winner](http://www.stat.ufl.edu/~winner/datasets.html) of the University of Florida. A [2013 study](http://digitalcommons.wku.edu/ijes/vol6/iss2/10/) in the International Journal of Exercise Science studied collegiate shot put athletes and examined the relation between strength and shot put distance. The population consists of 28 female collegiate athletes. Strength was measured by the the biggest amount (in kilograms) that the athlete lifted in the "1RM power clean" in the pre-season. The distance (in meters) was the athlete's personal best.

In [1]:
import polars as pl
import numpy as np
import hvplot.polars
path_data = '../../../assets/data/'
# POLARS_CHANGE: minimize is still the Data 8 helper.
from datascience import minimize

In [2]:
def standard_units(any_numbers):
    "Convert any array of numbers to standard units."
    # POLARS_CHANGE: .mean() and .std(ddof=0) work on a NumPy array and on a Polars Series (np.std is ddof=0).
    return (any_numbers - any_numbers.mean())/any_numbers.std(ddof=0)  

def correlation(t, x, y):
    return (standard_units(t.get_column(x))*standard_units(t.get_column(y))).mean()

def slope(table, x, y):
    r = correlation(table, x, y)
    return r * table.get_column(y).std(ddof=0)/table.get_column(x).std(ddof=0)

def intercept(table, x, y):
    a = slope(table, x, y)
    return table.get_column(y).mean() - a * table.get_column(x).mean()

def fit(table, x, y):
    """Return the height of the regression line at each x value."""
    a = slope(table, x, y)
    b = intercept(table, x, y)
    return a * table.get_column(x) + b

In [3]:
shotput = pl.read_csv(path_data + 'shotput.csv')

In [4]:
shotput

Weight Lifted,Shot Put Distance
f64,f64
37.5,6.4
51.5,10.2
61.3,12.4
61.3,13.0
63.6,13.2
…,…
123.6,16.8
125.8,17.0
119.1,18.2


In [5]:
shotput.hvplot.scatter(x='Weight Lifted', y='Shot Put Distance')

Scatterplot with 'Weight Lifted' on the x-axis and 'Shot Put Distance' on the y-axis. There is a clear positive correlation, with more data points that have higher x values.

That's not a football shaped scatter plot. In fact, it seems to have a slight non-linear component. But if we insist on using a straight line to make our predictions, there is still one best straight line among all straight lines.

Our formulas for the slope and intercept of the regression line, derived for football shaped scatter plots, give the following values.

In [6]:
slope(shotput, 'Weight Lifted', 'Shot Put Distance')

0.09834382159781997

In [7]:
intercept(shotput, 'Weight Lifted', 'Shot Put Distance')

5.959629098373954

Does it still make sense to use these formulas even though the scatter plot isn't football shaped? We can answer this by finding the slope and intercept of the line that minimizes the mse.

We will define the function `shotput_linear_mse` to take an arbirtary slope and intercept as arguments and return the corresponding mse. Then `minimize` applied to `shotput_linear_mse` will return the best slope and intercept.

In [8]:
def shotput_linear_mse(any_slope, any_intercept):
    x = shotput.get_column('Weight Lifted')
    y = shotput.get_column('Shot Put Distance')
    fitted = any_slope*x + any_intercept
    return ((y - fitted) ** 2).mean()

In [9]:
minimize(shotput_linear_mse)

array([ 0.09834382,  5.9596291 ])

These values are the same as those we got by using our formulas. To summarize:

**No matter what the shape of the scatter plot, there is a unique line that minimizes the mean squared error of estimation. It is called the regression line, and its slope and intercept are given by**

$$
\mathbf{\text{slope of the regression line}} ~=~ r \cdot
\frac{\text{SD of }y}{\text{SD of }x}
$$

$$
\mathbf{\text{intercept of the regression line}} ~=~
\text{average of }y ~-~ \text{slope} \cdot \text{average of }x
$$

In [10]:
fitted = fit(shotput, 'Weight Lifted', 'Shot Put Distance')
shotput.with_columns(fitted.alias('Best Straight Line')).hvplot.scatter(x='Weight Lifted', y=['Shot Put Distance', 'Best Straight Line'], color=['#002fa7', '#706700'])

The previous graph is reproduced with the addition of gold data points labeled 'Best Straight Line.' There is one data point per x value of the dark blue data points. We see that the gold points if connected would make a line with a positive slope. One notable deviation of the gold data points from the dark blue data points is for the smallest x value; it has an actual y value of just over 6 but a predicted y value of just under 10.

## Nonlinear Regression
The graph above reinforces our earlier observation that the scatter plot is a bit curved. So it is better to fit a curve than a straight line. The [study](http://digitalcommons.wku.edu/ijes/vol6/iss2/10/) postulated a quadratic relation between the weight lifted and the shot put distance. So let's use quadratic functions as our predictors and see if we can find the best one. 

We have to find the best quadratic function among all quadratic functions, instead of the best straight line among all straight lines. The method of least squares allows us to do this.

The mathematics of this minimization is complicated and not easy to see just by examining the scatter plot. But numerical minimization is just as easy as it was with linear predictors! We can get the best quadratic predictor by once again using `minimize`. Let's see how this works.

Recall that a quadratic function has the form

$$
f(x) ~=~ ax^2 + bx + c
$$

for constants $a$, $b$, and $c$.

To find the best quadratic function to predict distance based on weight lifted, using the criterion of least squares, we will first write a function that takes the three constants as its arguments, calculates the fitted values by using the quadratic function above, and then returns the mean squared error. 

The function is called `shotput_quadratic_mse`. Notice that the definition is analogous to that of `lw_mse`, except that the fitted values are based on a quadratic function instead of linear.

In [11]:
def shotput_quadratic_mse(a, b, c):
    x = shotput.get_column('Weight Lifted')
    y = shotput.get_column('Shot Put Distance')
    fitted = a*(x**2) + b*x + c
    return ((y - fitted) ** 2).mean()

We can now use `minimize` just as before to find the constants that minimize the mean squared error. 

In [12]:
best = minimize(shotput_quadratic_mse)
best

array([ -1.04004837e-03,   2.82708043e-01,  -1.53182103e+00])

Our prediction of the shot put distance for an athlete who lifts $x$ kilograms is about

$$
-0.00104x^2 ~+~ 0.2827x - 1.5318
$$

meters. For example, if the athlete can lift 100 kilograms, the predicted distance is 16.33 meters. On the scatter plot, that's near the center of a vertical strip around 100 kilograms.

In [13]:
(-0.00104)*(100**2) + 0.2827*100 - 1.5318

16.3382

Here are the predictions for all the values of `Weight Lifted`. You can see that they go through the center of the scatter plot, to a rough approximation.

In [14]:
x = shotput.to_series(0)
shotput_fit = best.item(0)*(x**2) + best.item(1)*x + best.item(2)

In [15]:
shotput.with_columns(shotput_fit.alias('Best Quadratic Curve')).hvplot.scatter(x='Weight Lifted', y=['Shot Put Distance', 'Best Quadratic Curve'], color=['#002fa7', '#706700'])

The previous graph is reproduced with different gold data points labeled 'Best Quadratic Line.' There is one data point per x value of the dark blue data points. We see that the gold points if connected would make a curve that increases rapidly until it begins to even out on the right hand side of the graph. The gold data points better follow the dark blue data points, including the data point with the smallest x value.

**Note:** We fit a quadratic here because it was suggested in the original study. But it is worth noting that at the rightmost end of the graph, the quadratic curve appears to be close to peaking, after which the curve will start going downwards. So we might not want to use this model for new athletes who can lift weights much higher than those in our data set.  